In [1]:
import pandas as pd

payment_df = pd.read_csv("../Data/cleaned/payment_and_value_cleaned.csv")
print("Shape:", payment_df.shape)
print("\nColumns:")
for col in payment_df.columns:
    print(f"  {col}")

Shape: (18372, 20)

Columns:
  facility_id
  facility_name
  address
  citytown
  state
  zip_code
  countyparish
  telephone_number
  payment_measure_id
  payment_measure_name
  payment_category
  denominator
  payment
  lower_estimate
  higher_estimate
  value_of_care_display_id
  value_of_care_display_name
  value_of_care_category
  start_date
  end_date


In [2]:
print("unique payment measures:")
print(payment_df[["payment_measure_id", "payment_measure_name"]].drop_duplicates().to_string())

print("\nunique value of care measures:")
print(payment_df[["value_of_care_display_id", "value_of_care_display_name"]].drop_duplicates().to_string())

unique payment measures:
  payment_measure_id                       payment_measure_name
0        PAYM_30_AMI          Payment for heart attack patients
1         PAYM_30_HF         Payment for heart failure patients
2         PAYM_30_PN             Payment for pneumonia patients
3   PAYM_90_HIP_KNEE  Payment for hip/knee replacement patients

unique value of care measures:
  value_of_care_display_id          value_of_care_display_name
0         MORT_PAYM_30_AMI  Value of Care Heart Attack measure
1          MORT_PAYM_30_HF  Value of Care Heart Failure measur
2          MORT_PAYM_30_PN     Value of Care Pneumonia measure
3    COMP_PAYM_90_HIP_KNEE  Value of Care hip/knee replacement


In [3]:
payment_df["payment"] = payment_df["payment"].replace("Not Available", None)
payment_df["payment"] = pd.to_numeric(payment_df["payment"], errors="coerce")

payment_df["value_of_care_category"] = payment_df["value_of_care_category"].replace("Not Available", None)
payment_df["payment_category"] = payment_df["payment_category"].replace("Not Available", None)

payment_pivot = payment_df.pivot_table(
    index="facility_id",
    columns="payment_measure_id",
    values="payment",
    aggfunc="first"
).reset_index()
payment_pivot.columns.name = None

category_pivot = payment_df.pivot_table(
    index="facility_id",
    columns="payment_measure_id",
    values="payment_category",
    aggfunc="first"
).reset_index()
category_pivot.columns.name = None
category_pivot.columns = ["facility_id"] + [f"{col}_category" for col in category_pivot.columns[1:]]

value_pivot = payment_df.pivot_table(
    index="facility_id",
    columns="value_of_care_display_id",
    values="value_of_care_category",
    aggfunc="first"
).reset_index()
value_pivot.columns.name = None

hospital_info = payment_df[["facility_id", "facility_name", "address",
                              "citytown", "state", "zip_code",
                              "countyparish", "telephone_number"]].drop_duplicates("facility_id")

df_final = hospital_info.merge(payment_pivot, on="facility_id", how="outer")
df_final = df_final.merge(category_pivot, on="facility_id", how="outer")
df_final = df_final.merge(value_pivot, on="facility_id", how="outer")

print("final shape:", df_final.shape)
print("duplicate facility_ids:", df_final["facility_id"].duplicated().sum())

final shape: (4593, 16)
duplicate facility_ids: 0
